# English Sentiment Model
Same idea as the Arabic one, just on the IMDB reviews dataset.
Expects `data/imdb_reviews.csv`.

In [ ]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from preprocessing_pipeline import preprocess_english


In [ ]:
df = pd.read_csv("data/imdb_reviews.csv")

text_col = "review" if "review" in df.columns else df.columns[0]
label_col = "sentiment" if "sentiment" in df.columns else df.columns[-1]

df = df[[text_col, label_col]].dropna()
print(df[label_col].value_counts())


## Preprocess

In [ ]:
df["clean_text"] = df[text_col].apply(preprocess_english)
df = df[df["clean_text"].str.len() > 0]


## Split, vectorize, train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df[label_col], test_size=0.2, random_state=42, stratify=df[label_col]
)

vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=15000)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)


In [ ]:
preds = model.predict(X_test_vec)
print("accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))


## Save weights

In [ ]:
with open("English_model_weights.pkl", "wb") as f:
    pickle.dump({"vectorizer": vectorizer, "model": model}, f)

print("saved English_model_weights.pkl")
